# Task 2: Reproduction on Wine Dataset
**Paper:** Gaussian Mixture Model with Local Consistency — AAAI 2010  
**Student:** Bhavishya Sahay | Roll No: 230071

**Contribution being reproduced:** The core LCGMM clustering algorithm (Table 2 metric: clustering accuracy AC).  
**Evaluation metric:** Clustering Accuracy (AC), Equation (15) from the paper, using Hungarian algorithm for label mapping.

## Task 2.1 — Dataset Selection

I use the **Wine dataset** (UCI / sklearn) for reproducing LCGMM. It has 178 samples, 13 continuous chemical features, and 3 ground-truth classes corresponding to three cultivars of wine from Italy. This makes it a natural testbed: it is a **multi-class clustering problem** with continuous features and known ground truth, matching the problem type studied in the paper. The moderate number of features (13) is tractable for the full LCGMM EM loop on a CPU. The Wine dataset likely has simpler geometric structure than the paper's MNIST (784-dim) or Yale Face (4096-dim) datasets, so we expect LCGMM to show smaller relative improvement over GMM — this is an honest limitation. All 13 features are standardised to zero mean and unit variance before clustering.

In [17]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# REPRODUCIBILITY: Set all random seeds here
# ============================================================
SEED = 42
np.random.seed(SEED)

# ============================================================
# HYPERPARAMETERS: All defined in one place
# ============================================================
K = 3          # Number of clusters
P_NEIGHBORS = 10   # k-NN graph parameter p
LAMBDA = 0.1   # Regularisation strength lambda
MAX_ITER = 100 # EM max iterations
TOL = 1e-4     # Convergence tolerance
REG = 1e-4     # Covariance regularisation (numerical stability)

print("All imports and hyperparameters set.")
print(f"K={K}, p={P_NEIGHBORS}, lambda={LAMBDA}, max_iter={MAX_ITER}")

All imports and hyperparameters set.
K=3, p=10, lambda=0.1, max_iter=100


The cell above imports all required libraries and defines all hyperparameters in one place for reproducibility. `SEED=42` is used throughout.

In [18]:
# Load the Wine dataset
data = load_wine()
X_raw, y_true = data.data, data.target

# Standardise features (zero mean, unit variance)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

N, D = X.shape
print(f"Dataset: Wine (UCI)")
print(f"Samples: {N}, Features: {D}, Classes: {len(np.unique(y_true))}")
print(f"Class distribution: {np.bincount(y_true)}")

Dataset: Wine (UCI)
Samples: 178, Features: 13, Classes: 3
Class distribution: [59 71 48]


The Wine dataset has 178 samples and 13 features across 3 classes. Features are standardised using `StandardScaler` so that no single feature dominates distance calculations in the kNN graph.

In [19]:
# Clustering Accuracy metric — Eq.(15) from the paper
# Uses the Kuhn-Munkres (Hungarian) algorithm for optimal label mapping
def clustering_accuracy(y_true, y_pred):
    n = max(y_true.max(), y_pred.max()) + 1
    cost = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        cost[t][p] += 1
    row, col = linear_sum_assignment(-cost)
    return cost[row, col].sum() / len(y_true)

print("clustering_accuracy() defined — matches Eq.(15) from the paper.")

clustering_accuracy() defined — matches Eq.(15) from the paper.


The clustering accuracy metric uses the **Kuhn-Munkres (Hungarian) algorithm** to find the optimal bijective mapping between predicted cluster labels and true class labels. This matches Equation (15) in the paper exactly.

## Task 2.2 — LCGMM Implementation

In [20]:
# Step 1: Build k-NN Graph — Eq.(4)
# W_ij = 1 if x_i is in p-NN of x_j OR x_j is in p-NN of x_i, else 0
def build_knn_graph(X, p=P_NEIGHBORS):
    from sklearn.neighbors import kneighbors_graph
    G = kneighbors_graph(X, n_neighbors=p, mode='connectivity', include_self=False)
    G_arr = G.toarray()
    return np.maximum(G_arr, G_arr.T)  # symmetrise

W = build_knn_graph(X, P_NEIGHBORS)
print(f"Graph built: {W.shape}, edges={int(W.sum()//2)}")

Graph built: (178, 178), edges=1231


**Step 1 (Eq. 4):** Constructs the symmetric k-NN adjacency matrix W. Each point xi is connected to its p=10 nearest neighbours. The graph is symmetrised so that if xi is in xj's neighbourhood, the edge (i,j) is included in both directions. This graph encodes the manifold structure of the data.

In [21]:
# Helper: log-probability under a single Gaussian N(x | mu, Sigma)
def log_gaussian(x, mu, Sigma):
    d = len(mu)
    diff = x - mu
    try:
        sign, logdet = np.linalg.slogdet(Sigma)
        if sign <= 0:
            return -1e10
        inv_S = np.linalg.inv(Sigma)
        return -0.5 * (d * np.log(2 * np.pi) + logdet + diff @ inv_S @ diff)
    except Exception:
        return -1e10

print("log_gaussian() defined.")

log_gaussian() defined.


Helper function computing the log-density of a Gaussian. Used in the E-step (Eq. 7) to compute the unnormalised log-responsibility for each (point, component) pair.

In [22]:
# ============================================================
# LCGMM: Full Implementation
# ============================================================
def run_lcgmm(X, K=K, p=P_NEIGHBORS, lam=LAMBDA, max_iter=MAX_ITER, tol=TOL, reg=REG):
    """
    Locally Consistent Gaussian Mixture Model (LCGMM)
    Liu et al., AAAI 2010.
    
    Returns: (labels, posterior_matrix R)
    """
    N, D = X.shape
    W = build_knn_graph(X, p)
    rows, cols = np.where(W > 0)  # sparse iteration over edges

    # --- Initialise with standard GMM ---
    gmm_init = GaussianMixture(n_components=K, random_state=SEED, reg_covar=reg)
    gmm_init.fit(X)
    pi = gmm_init.weights_.copy()
    mu = gmm_init.means_.copy()
    Sigma = gmm_init.covariances_.copy() + reg * np.eye(D)

    prev_obj = -np.inf
    for iteration in range(max_iter):
        # ---- E-step: compute posteriors P(c_k | x_i) — Eq.(7) ----
        log_resp = np.zeros((N, K))
        for k in range(K):
            for i in range(N):
                log_resp[i, k] = np.log(pi[k] + 1e-300) + log_gaussian(X[i], mu[k], Sigma[k])
        log_norm = logsumexp(log_resp, axis=1, keepdims=True)
        R = np.exp(log_resp - log_norm)   # R[i,k] = P(c_k | x_i)

        # ---- M-step: update pi — Eq.(9) ----
        Nk = R.sum(axis=0) + 1e-10
        pi = Nk / N

        for k in range(K):
            rk = R[:, k]  # posterior for component k

            # Update mu — Eq.(13)
            mu_std = (rk[:, None] * X).sum(0) / Nk[k]
            reg_mu = np.zeros(D)
            for i, j in zip(rows, cols):
                reg_mu += (rk[i] - rk[j]) * (X[i] - X[j])
            mu[k] = mu_std - lam * reg_mu / (2 * Nk[k])

            # Update Sigma — Eq.(14)
            diff = X - mu[k]
            S_std = (rk[:, None, None] * diff[:, :, None] * diff[:, None, :]).sum(0) / Nk[k]
            reg_S = np.zeros((D, D))
            for i, j in zip(rows, cols):
                Si = np.outer(X[i] - mu[k], X[i] - mu[k])
                Sj = np.outer(X[j] - mu[k], X[j] - mu[k])
                reg_S += (rk[i] - rk[j]) * (Si - Sj)
            Sigma[k] = S_std - lam * reg_S / (2 * Nk[k]) + reg * np.eye(D)

        # Convergence check on regularised log-likelihood — Eq.(6)
        obj = log_norm.sum()
        if abs(obj - prev_obj) < tol:
            print(f"  Converged at iteration {iteration+1}")
            break
        prev_obj = obj

    return R.argmax(axis=1), R

print("run_lcgmm() defined.")

run_lcgmm() defined.


**Full LCGMM implementation.** The E-step (Eq. 7) is identical to standard GMM. The M-step updates for μₖ (Eq. 13) and Σₖ (Eq. 14) each contain an additional regularisation correction that sums over all edges (i,j) in the k-NN graph, weighted by the difference in posterior probabilities [P(cₖ|xᵢ) − P(cₖ|xⱼ)]. This is the core novelty of LCGMM over standard GMM.

In [23]:
# Run LCGMM and baselines
print("Running LCGMM...")
lcgmm_labels, lcgmm_R = run_lcgmm(X)
lcgmm_acc = clustering_accuracy(y_true, lcgmm_labels)

gmm = GaussianMixture(n_components=K, random_state=SEED)
gmm.fit(X)
gmm_acc = clustering_accuracy(y_true, gmm.predict(X))

km = KMeans(n_clusters=K, random_state=SEED, n_init=10)
km.fit(X)
km_acc = clustering_accuracy(y_true, km.labels_)

sc = SpectralClustering(n_clusters=K, random_state=SEED, affinity='nearest_neighbors', n_neighbors=P_NEIGHBORS)
sc_acc = clustering_accuracy(y_true, sc.fit_predict(X))

print(f"\nResults on Wine Dataset:")
print(f"  LCGMM : {lcgmm_acc*100:.1f}%")
print(f"  GMM   : {gmm_acc*100:.1f}%")
print(f"  K-Means: {km_acc*100:.1f}%")
print(f"  NCut  : {sc_acc*100:.1f}%")

Running LCGMM...
  Converged at iteration 22

Results on Wine Dataset:
  LCGMM : 93.8%
  GMM   : 96.6%
  K-Means: 96.6%
  NCut  : 96.1%


All four methods are run on the same standardised Wine data. The clustering accuracy is computed using the Hungarian-mapping metric (Eq. 15) for fair comparison.

## Task 2.3 — Results, Comparison and Reproducibility

In [24]:
# Visualisation: bar chart + PCA scatter — save to partB/results/
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
methods = ['LCGMM', 'GMM', 'K-Means', 'NCut']
accs = [lcgmm_acc*100, gmm_acc*100, km_acc*100, sc_acc*100]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
ax = axes[0]
bars = ax.bar(methods, accs, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylim(0, 110)
ax.set_ylabel('Clustering Accuracy (%)', fontsize=12)
ax.set_title('Clustering Accuracy on Wine Dataset\n(LCGMM vs Baselines)', fontsize=12, fontweight='bold')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{acc:.1f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# PCA scatter
pca = PCA(n_components=2)
X2 = pca.fit_transform(X)
ax2 = axes[1]
sc_colors = ['#2196F3', '#FF5722', '#4CAF50']
for k in range(K):
    mask = lcgmm_labels == k
    ax2.scatter(X2[mask,0], X2[mask,1], c=sc_colors[k], label=f'Cluster {k}',
                alpha=0.75, edgecolors='k', linewidth=0.3, s=60)
for k in range(K):
    mask = y_true == k
    ax2.scatter(X2[mask,0], X2[mask,1], marker='x', c='black', alpha=0.25, s=25)
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=11)
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=11)
ax2.set_title('LCGMM Clusters on Wine (PCA)\n(coloured=predicted, x=true labels)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
plt.tight_layout()
plt.savefig('/Users/bhavishyamac/Desktop/230071-midesm/partB/results/task2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to partB/results/task2_comparison.png")

Saved to partB/results/task2_comparison.png


The bar chart compares all four methods. The PCA scatter plot shows LCGMM's predicted cluster assignments (colour) alongside the true class labels (x markers). Points where the colour and x marker class disagree represent misclassifications.

In [25]:
# Discussion and Reproducibility Checklist
print("=" * 60)
print("RESULT COMPARISON")
print("=" * 60)
print(f"Our LCGMM on Wine:  {lcgmm_acc*100:.1f}%")
print(f"Paper LCGMM (Table 2, best): ~73.6% (MNIST), ~100% (Cloud)")
print(f"Our GMM baseline:   {gmm_acc*100:.1f}%")
print()
print("NOTE: The paper's Table 2 uses MNIST (10 classes, 10000 samples).")
print("We use Wine (3 classes, 178 samples) — a simpler, different domain.")
print()
print("=" * 60)
print("REPRODUCIBILITY CHECKLIST")
print("=" * 60)
checklist = [
    "Random seeds set at top of notebook (SEED=42)",
    "All dependencies in requirements.txt with versions",
    "Notebook runs top-to-bottom without errors",
    "Dataset loading: sklearn.datasets.load_wine() — no manual steps",
    "All hyperparameters defined in one place (top of notebook)",
]
for item in checklist:
    print(f"  [x] {item}")

RESULT COMPARISON
Our LCGMM on Wine:  93.8%
Paper LCGMM (Table 2, best): ~73.6% (MNIST), ~100% (Cloud)
Our GMM baseline:   96.6%

NOTE: The paper's Table 2 uses MNIST (10 classes, 10000 samples).
We use Wine (3 classes, 178 samples) — a simpler, different domain.

REPRODUCIBILITY CHECKLIST
  [x] Random seeds set at top of notebook (SEED=42)
  [x] All dependencies in requirements.txt with versions
  [x] Notebook runs top-to-bottom without errors
  [x] Dataset loading: sklearn.datasets.load_wine() — no manual steps
  [x] All hyperparameters defined in one place (top of notebook)


## Interpretation of Results

Our LCGMM achieves **93.8%** accuracy on Wine, compared to GMM's **96.6%** — LCGMM is slightly lower, not higher, than the vanilla baseline. This is consistent with the paper's own observation that LCGMM does not always outperform GMM (e.g., on Waveform, GMM 76.3% > LCGMM 75.3%).

**Reasons for the gap:**
1. **Well-separated Gaussian clusters:** The Wine dataset has three chemically distinct clusters that are approximately Gaussian in standardised 13-D feature space. In this setting, standard GMM already captures the cluster structure well, and the kNN regulariser may pull cluster boundaries in slightly suboptimal directions.
2. **Dataset scale:** The paper's most striking LCGMM gains (e.g., Chart: 70% vs 56.8%, Vowel: 36.6% vs 31.9%) come from datasets with stronger manifold structure. Wine's 13-D chemical features do not exhibit the same kind of intrinsic manifold geometry as handwritten digit images.
3. **Lambda sensitivity:** At λ=0.1 the regulariser has a small but non-negligible effect. Our sensitivity analysis (Task 3.2) shows that very small λ (~0.001–0.01) slightly outperforms even GMM on this dataset, suggesting that moderate regularisation helps but the default λ=0.1 slightly over-regularises for Wine.

The performance difference is an honest observation, not a failure of the implementation — the paper itself acknowledges that LCGMM ranks second on some datasets.
